# 03: Proteomic Clinical Index Scoring

This notebook turns a proteomics classifier into a **clinically readable index layer**.

The goal is to demonstrate the move from model output to product output:

- calibrated probability → 0–100 proteomic index
- index score → red/amber/green risk band
- risk band + confidence → clinician-review gate
- model coefficients → top protein drivers

This mirrors how a clinical proteomics platform would convert biological signal into a structured index-level result rather than stopping at AUROC.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from proteomics_index.modeling import make_baseline_models, binary_metrics
from proteomics_index.scoring import (
    compute_index_table,
    linear_model_feature_contributions,
)
from proteomics_index.confidence import (
    missingness_penalty,
    confidence_score_from_signals,
    confidence_label,
)

DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

## 1. Load ML-ready proteomics matrix

In [2]:
X_raw = pd.read_csv(DATA_PROCESSED / "X_proteomics.csv")
X = X_raw.set_index("SampleID") if "SampleID" in X_raw.columns else X_raw
y_raw = pd.read_csv(DATA_PROCESSED / "y_severe_outcome.csv")
y = y_raw.set_index("SampleID")["SevereOutcome"].loc[X.index] if "SampleID" in y_raw.columns else y_raw.squeeze("columns")
metadata = pd.read_csv(DATA_PROCESSED / "metadata_model.csv")
metadata = metadata.set_index("SampleID").loc[X.index]

print("X shape:", X.shape)
print("Event rate:", round(float(y.mean()), 3))
X.head()

X shape: (96, 240)
Event rate: 0.167


,Protein_000,Protein_001,Protein_002,Protein_003,Protein_004,Protein_005,Protein_006,Protein_007,Protein_008,Protein_009,Protein_010,Protein_011,Protein_012,Protein_013,Protein_014,Protein_015,Protein_016,Protein_017,Protein_018,Protein_019,Protein_020,Protein_021,Protein_022,Protein_023,Protein_024,Protein_025,Protein_026,Protein_027,Protein_028,Protein_029,Protein_030,Protein_031,Protein_032,Protein_033,Protein_034,Protein_035,Protein_036,Protein_037,Protein_038,Protein_039,...,Protein_200,Protein_201,Protein_202,Protein_203,Protein_204,Protein_205,Protein_206,Protein_207,Protein_208,Protein_209,Protein_210,Protein_211,Protein_212,Protein_213,Protein_214,Protein_215,Protein_216,Protein_217,Protein_218,Protein_219,Protein_220,Protein_221,Protein_222,Protein_223,Protein_224,Protein_225,Protein_226,Protein_227,Protein_228,Protein_229,Protein_230,Protein_231,Protein_232,Protein_233,Protein_234,Protein_235,Protein_236,Protein_237,Protein_238,Protein_239
SampleID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
S001,8.2924,7.1298,9.2307,8.4142,6.8576,7.8754,8.1206,7.9044,6.8970,NaN,9.0807,7.9216,6.9082,7.4631,8.6958,9.0429,8.1175,8.3892,6.7859,9.0286,7.6023,8.7397,8.4008,7.5012,7.3667,8.6641,8.4824,8.3096,10.3320,8.8504,6.6058,9.7943,7.4886,8.3538,5.5277,7.8420,8.6632,7.5954,8.0314,7.6912,...,6.4763,7.9254,6.6147,8.1034,8.6195,8.5023,7.8631,7.8484,8.4330,8.9067,8.4087,7.7183,8.5174,9.1995,7.9541,7.3107,9.5529,6.2561,8.6144,6.8438,6.6846,7.8132,7.6611,6.7127,8.3446,7.1964,6.7783,7.2810,6.8521,8.0290,7.7393,8.0597,7.2694,7.9637,5.6277,8.2776,NaN,9.2960,9.4927,8.2496
S002,8.1125,8.4070,8.5497,9.7698,9.2736,7.9969,8.8650,8.0162,8.7703,7.8049,7.6619,7.9814,9.6609,9.0135,8.9213,9.8956,8.1680,8.8768,8.8718,9.3833,9.3286,8.6106,8.9157,10.4981,8.2624,8.8909,8.5714,10.4943,8.3999,9.7809,7.1367,8.7649,8.3520,9.0461,9.3189,7.8824,8.6445,9.4800,7.2816,8.2671,...,7.8800,8.5204,9.3802,8.9423,9.1345,8.0523,7.5759,9.6420,8.6772,8.9969,7.8120,8.7282,8.6663,7.3287,9.8016,7.3847,7.9933,8.4165,11.2321,8.1310,8.8122,9.0531,8.0281,8.5183,8.9137,8.9722,9.0785,8.1686,7.3991,8.1229,8.0117,8.6719,7.9297,7.9671,8.4802,8.1862,7.7431,8.9586,8.1789,9.3435
S003,8.1130,8.9215,8.5633,8.0311,9.5636,8.5631,7.5421,9.9396,8.4572,9.3918,9.8291,8.4636,9.1724,8.4980,7.5791,8.5489,9.4604,NaN,8.2875,8.9317,9.1267,8.5768,9.5350,9.5583,9.1581,9.8982,9.2141,7.0354,10.1002,8.4141,9.6909,10.4768,9.5678,7.6772,8.9666,8.3058,8.0040,7.9146,7.9186,7.2125,...,9.0384,9.8178,7.9889,9.3615,7.0301,7.0244,8.0128,7.7870,NaN,7.3609,10.1988,8.1935,8.6932,7.7038,8.1475,8.7596,8.1045,8.3570,8.3181,8.1717,8.9955,7.5470,9.2107,8.4931,7.1985,8.2937,11.3352,9.8232,8.1458,8.2972,8.0054,7.2266,8.6324,9.2495,7.9455,7.6251,7.1184,7.8506,6.5896,7.8499
S004,7.4966,7.8415,6.9313,7.9216,8.5251,10.1793,7.0994,8.4568,8.3548,8.6983,7.4569,8.5136,9.3065,7.9794,7.9712,8.9884,6.8756,9.9894,8.7170,8.6490,8.4723,8.8531,9.3345,8.7636,7.6412,5.9970,7.9451,8.8707,8.1107,8.7371,8.2184,8.2510,8.3606,7.7051,8.6082,9.7774,7.0588,7.0925,NaN,8.1261,...,7.8624,9.1400,8.0502,9.6565,9.3938,7.5112,7.8696,9.1423,7.0827,9.1023,7.8117,9.7076,8.2968,9.1064,8.5841,8.2466,7.1579,9.0677,9.3937,8.6405,10.4637,7.2630,7.4646,6.7181,9.0269,8.9170,9.5329,7.8065,9.2090,7.1936,8.3512,8.8011,8.3190,8.7063,8.4781,7.8913,8.2284,9.1607,8.3555,9.2828
S005,7.1772,7.9123,8.2787,10.8516,8.4385,7.5880,8.1722,7.5648,9.0011,7.8961,6.8824,9.4176,8.1771,6.4765,NaN,6.4747,8.3517,8.0406,8.5542,9.5917,8.4298,7.8528,10.1355,8.9538,8.2227,8.2576,7.8137,7.1645,7.9417,7.0395,9.9092,8.7709,NaN,8.9961,8.5649,8.9004,8.4450,9.5271,7.5025,8.2676,...,8.2108,9.1689,NaN,8.8473,8.0189,8.3869,8.3620,6.7710,8.2494,7.7719,7.5105,8.8808,7.4779,7.5096,7.7666,9.2938,9.5467,9.0139,7.8747,8.4278,9.0363,8.5293,7.3136,9.0623,8.4420,6.8681,10.3570,6.9926,7.2588,7.5483,7.6938,9.4482,7.8385,6.7535,9.0102,8.2013,6.2505,7.7845,8.8049,8.6444


## 2. Fit a sparse proteomics model

Elastic-net logistic regression is a practical baseline for small-sample, high-dimensional proteomics because it regularises the feature space and can produce interpretable coefficients.

In [3]:
models = make_baseline_models(k_features=40, random_state=42)
model = models["ElasticNet_LogReg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
metrics = binary_metrics(y_test, y_prob)
metrics

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


{'AUROC': 0.8875,
 'AUPRC': 0.7708333333333333,
 'Brier': 0.07249468707473243,
 'Sensitivity': np.float64(0.75),
 'Specificity': np.float64(0.95)}

## 3. Convert probability into a clinical index

A clinical index is easier for clinicians to consume than raw probability. Here we map probability to a 0–100 index and assign transparent bands:

- Green: lower index
- Amber: intermediate index
- Red: elevated index

The thresholds are demonstrative and must be clinically validated before real use.

In [4]:
miss = missingness_penalty(X_test)
conf_scores = confidence_score_from_signals(y_prob, miss)
conf_labels = confidence_label(conf_scores)

index_table = compute_index_table(
    sample_ids=X_test.index,
    probabilities=y_prob,
    confidence_labels=conf_labels,
    confidence_scores=conf_scores,
)
index_table = index_table.set_index("sample_id").loc[X_test.index].reset_index().rename(columns={"SampleID": "sample_id"})
index_table.to_csv(REPORTS / "patient_index_examples.csv", index=False)
index_table.head(10)

,sample_id,probability,index_score,risk_band,confidence_label,confidence_score,decision_gate,ci_lower,ci_upper
0,S093,0.574282,57,Amber,Low,0.260542,Clinician review required: low confidence,None,None
1,S089,0.151596,15,Green,Moderate,0.639202,Routine clinician review,None,None
2,S061,0.732203,73,Red,Moderate,0.453678,Clinician review required: elevated index,None,None
3,S015,0.030742,3,Green,High,0.898235,Routine clinician review,None,None
4,S027,0.055986,6,Green,High,0.838277,Routine clinician review,None,None
5,S073,0.689811,69,Red,Low,0.398619,Clinician review required: low confidence,None,None
6,S091,0.005481,1,Green,High,0.965993,Routine clinician review,None,None
7,S026,0.137407,14,Green,Moderate,0.669480,Routine clinician review,None,None
8,S065,0.052458,5,Green,High,0.841783,Routine clinician review,None,None
9,S018,0.162861,16,Green,Moderate,0.624275,Routine clinician review,None,None


## 4. Explain a single patient/sample index result

In [5]:
example_sample_id = index_table.sort_values("index_score", ascending=False).iloc[0]["sample_id"]
example_row = X_test.loc[example_sample_id]
contrib = linear_model_feature_contributions(
    fitted_pipeline=model,
    sample=example_row,
    feature_names=list(X.columns),
    top_n=10,
)
contrib.to_csv(REPORTS / "example_patient_top_protein_drivers.csv", index=False)
print("Example sample:", example_sample_id)
index_table[index_table["sample_id"] == example_sample_id]

Example sample: S076


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


,sample_id,probability,index_score,risk_band,confidence_label,confidence_score,decision_gate,ci_lower,ci_upper
12,S076,0.913485,91,Red,High,0.773482,Clinician review required: elevated index,None,None


In [6]:
contrib

,feature,coefficient,contribution,direction,absolute_contribution
0,Protein_011,0.664703,2.488534,increases index,2.488534
1,Protein_013,0.559434,0.364726,increases index,0.364726
2,Protein_227,0.311856,0.349370,increases index,0.349370
3,Protein_012,-0.406345,0.331345,increases index,0.331345
4,Protein_017,0.136607,0.301861,increases index,0.301861
5,Protein_016,0.473914,0.301579,increases index,0.301579
6,Protein_172,0.219106,-0.185021,decreases index,0.185021
7,Protein_014,0.064857,0.143896,increases index,0.143896
8,Protein_010,0.195799,0.142699,increases index,0.142699
9,Protein_019,0.219088,0.126960,increases index,0.126960


## 5. What this notebook demonstrates

This notebook shows the **scoring-methodology layer**: a model output becomes a traceable, clinician-readable proteomic severity index with risk banding, confidence labels, and protein-level explanation hooks.